In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import argparse
import scanpy as sc
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import sys
sys.path.append("..")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
import anndata as ad
import pandas as pd
from collections import Counter
import anndata
from sklearn.model_selection import train_test_split
from lib_vqvae.vqvae import VQVAE
from lib_vqvae.dataset import *

In [ ]:
# change to your file path
adata = sc.read_h5ad("/data/dya/CCC/CellNEST-main/HD_data_halved/count_area_filtered_adata_p75_1.h5ad")
adata.var_names_make_unique()

In [ ]:
# ====== Model Configuration & Initialization for Cell Tokenization (VQ-VAE) ======
import warnings
warnings.filterwarnings("ignore")

import os
import torch
import argparse

# Argument parser for model training & inference
parser = argparse.ArgumentParser(description="VQ-VAE for Spatial Transcriptomics Cell Tokenization")
parser.add_argument("--train_batch_size", type=int, default=64)
parser.add_argument("--eval_batch_size", type=int, default=2000)
parser.add_argument("--encoder_hidden_dim", type=list, default=[1600,1024,800])
parser.add_argument("--decoder_hidden_dim", type=list, default=[800,1024,1600])
parser.add_argument("--codebook_dim", type=int, default=8)
parser.add_argument("--n", type=int, default=32)
parser.add_argument("--n_codebooks", type=int, default=256)
parser.add_argument("--gamma", type=float, default=5)
parser.add_argument("--beta", type=float, default=10)
parser.add_argument('--weight_decay', type=float, default=5e-4)
parser.add_argument("--lr", type=float, default=1e-4)
parser.add_argument('--max_iter', type=int, default=20)
parser.add_argument('--seed', type=int, default=2024)
parser.add_argument('--gpu', type=int, default=0)
parser.add_argument("--dropout_rate", type=float, default=0.2)
parser.add_argument('--mode', type=str, default="vqvae")
parser.add_argument('--hvg', type=int, default=5000)

args = parser.parse_args([])

# Set computing device (GPU/CPU)
if torch.cuda.is_available():
    args.device = 'cuda'
    torch.cuda.set_device(args.gpu)
else:
    args.device = 'cpu'
print(f"Using device: {args.device}")

# Auto-detect project root & define model path (OPENSOURCE STANDARD)
ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
model_path = os.path.join(ROOT_DIR, "model", "cell_tokenizer", "model.pth")

# Initialize VQ-VAE model
data_dim = 5000
vae = VQVAE(
    data_dim, 
    args.n, 
    args.codebook_dim, 
    args.n_codebooks, 
    args.encoder_hidden_dim, 
    args.decoder_hidden_dim, 
    args.beta, 
    args.gamma, 
    args.dropout_rate
).to(args.device)

# Load pre-trained model weights from relative path
checkpoint = torch.load(model_path, map_location=args.device)
vae.load_state_dict(checkpoint['model'])
vae.eval()
print(f"Pre-trained VQ-VAE model loaded & set to evaluation mode: {model_path}")

In [ ]:
var_names_df = pd.read_csv("/nfs/public/xcx/CellTok/data/5000_gene_names.csv")
gene_names = var_names_df['gene'].tolist()

existing_genes = adata.var_names.isin(gene_names)
adata_filtered = adata[:, existing_genes]

existing_gene_names = adata.var_names[existing_genes].tolist()
result_df = pd.DataFrame(index=adata.obs_names, columns=gene_names)
result_df[:] = 0

if issparse(adata_filtered.X):
    adata_filtered_df = adata_filtered.X.toarray()
else:
    adata_filtered_df = adata_filtered.X

result_df.loc[:, existing_gene_names] = adata_filtered_df

new_adata = sc.AnnData(X=result_df.values.astype(np.float32), obs=adata.obs, var=pd.DataFrame(index=gene_names))
new_adata.var['feature_name'] = gene_names
new_adata.var_names = gene_names

new_adata

In [ ]:
adata = new_adata
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
vae.get_adata_codebook_index(adata, mode=args.mode, device=args.device)

In [ ]:
del adata.X
del adata.obsm['recon_x']
del adata.obsm['latent']

In [ ]:
adata_code=adata

In [ ]:
code_index = adata_code.obsm["code_index"]  

cell_codes = {}
for i in range(code_index.shape[0]):
    cell_codes[i] = code_index[i].tolist()  

In [ ]:
# Load constructed cell communication dataset
result_df = pd.read_csv(os.path.join(args.data_from, "combined_communication_pairs.csv"))

In [ ]:
# Construct instruction tuning dataset format for Hugging Face
def create_hf_format(row):
    # Retrieve token codes for both cells
    cell1_codes = cell_codes[row['cell_index_1']]
    cell2_codes = cell_codes[row['cell_index_2']]
    
    # Convert numerical codes into special token strings
    cell1_tokens = [f'<code_{code}>' for code in cell1_codes]
    cell2_tokens = [f'<code_{code}>' for code in cell2_codes]
    
    # Build input instruction text
    input_text = (
        f"Cell1: <cell_token_start>{''.join(cell1_tokens)}<cell_token_end>, "
        f"Cell2: <cell_token_start>{''.join(cell2_tokens)}<cell_token_end>, "
        f"determine whether there is intercellular communication between them."
    )
    
    # Build answer label text
    output_text = "Yes" if row['has_communication'] else "No"
    
    # Return standardized dataset dictionary
    return {
        'input_text': input_text,
        'output_text': output_text,
        'cell1': row['cell_index_1'],
        'cell2': row['cell_index_2'],
        'has_communication': row['has_communication']
    }

In [ ]:
hf_data = []
for _, row in result_df.iterrows():
    hf_data.append(create_hf_format(row))

hf_df = pd.DataFrame(hf_data)

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer)

# load the tokenizer of Qwen
tokenizer = AutoTokenizer.from_pretrained("/stor/dya/trajectory/tokenizer/tokenizer/")

In [ ]:
def preprocess_function_tokenized(examples):
    batch_size = len(examples[text_column])
    targets = [str(x) for x in examples[label_column]]
    model_inputs = tokenizer(examples[text_column])
    labels = tokenizer(targets, add_special_tokens=False)  # don't add bos token because we concatenate with inputs
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i] + [tokenizer.eos_token_id]
        model_inputs["input_ids"][i] = sample_input_ids + label_input_ids
        labels["input_ids"][i] = [-100] * len(sample_input_ids) + label_input_ids
        model_inputs["attention_mask"][i] = [1] * len(model_inputs["input_ids"][i])
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i]
        model_inputs["input_ids"][i] = [tokenizer.pad_token_id] * (
            max_length - len(sample_input_ids)
        ) + sample_input_ids
        model_inputs["attention_mask"][i] = [0] * (max_length - len(sample_input_ids)) + model_inputs[
            "attention_mask"
        ][i]
        labels["input_ids"][i] = [-100] * (max_length - len(sample_input_ids)) + label_input_ids
        model_inputs["input_ids"][i] = model_inputs["input_ids"][i][:max_length]
        model_inputs["attention_mask"][i] = model_inputs["attention_mask"][i][:max_length]
        labels["input_ids"][i] = labels["input_ids"][i][:max_length]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

text_column = "input_text"
label_column = "output_text"
max_length = 200
batch_size = 30

In [ ]:
from datasets import Dataset
hf_dataset = Dataset.from_pandas(hf_df)
hf_dataset

In [ ]:
processed_datasets = hf_dataset.map(
    preprocess_function_tokenized,
    batched=True,
    num_proc=10,
    load_from_cache_file=True,
    desc="Running tokenizer on dataset",
)
processed_datasets